# ROS02 · ROS 2 services and parameters

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/03_ros2/ROS02_ros_2_services_and_parameters.ipynb)

**Track:** ROS 2 · **Skill axis (sheet):** Real-time and Linux · **Time:** about 55 min

**Skill Hub concepts:** ROS 2 services and parameters

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each node task has an optional C++ (rclcpp) cell after it. The grader builds it with colcon and runs the same checker against it.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("ROS02")

## Map: three ways nodes talk, and where configuration lives

| Mechanism | Pattern | Use it for | Avoid for |
|---|---|---|---|
| Topic | publish/subscribe, fire and forget | sensor streams, state, setpoints | anything needing a reply |
| Service | request/response, one at a time | quick commands with a result (`open gripper`, `reset odometry`) | long jobs, high-rate data |
| Action | goal, feedback, result, cancel (lab ROS04) | long jobs (`move to pose`, `home axis`) | |
| Parameter | per-node typed key/value, settable at runtime | gains, limits, frame names | data that changes every cycle |

Every node automatically serves six parameter services: `describe_parameters`, `get_parameter_types`,
`get_parameters`, `list_parameters`, `set_parameters`, `set_parameters_atomically`.
A set-parameters callback can **reject** a value (return `SetParametersResult(successful=False, reason=...)`).

```
 ros2 service call /gripper/set_closed std_srvs/srv/SetBool "{data: true}"
 ros2 param set /gripper max_force 35.0        ros2 param get /gripper max_force       ros2 param dump /gripper
```

In [ ]:
from roboprep import ros
ros.ros("ros2 interface show std_srvs/srv/SetBool")

## Task 1: the gripper server
Node `gripper`, service `/gripper/set_closed` (`std_srvs/srv/SetBool`). Parameter `max_force` (double, default 20.0)
that rejects values outside [0, 50]. Response: `success=True`, message `"closed with 20.0 N"` (current force,
one decimal) when `data` is true, `"opened"` when false.

In [ ]:
%%writefile gripper_server.py
import rclpy
from rcl_interfaces.msg import SetParametersResult
from rclpy.executors import ExternalShutdownException
from rclpy.node import Node
from std_srvs.srv import SetBool


class Gripper(Node):
    def __init__(self):
        super().__init__("gripper")
        pass  # TODO: declare max_force, register the validation callback, create the service

    def on_params(self, params):
        return SetParametersResult(successful=True)  # TODO: reject out-of-range max_force

    def on_set(self, req, resp):
        return resp  # TODO


def main():
    rclpy.init()
    node = Gripper()
    try:
        rclpy.spin(node)
    except (KeyboardInterrupt, ExternalShutdownException):
        pass


if __name__ == "__main__":
    main()

## Task 2: a client
`python client.py close` (or `open`) calls the service, waits for the answer and prints `response.message`.

In [ ]:
%%writefile client.py
import sys
import rclpy
from rclpy.node import Node
from std_srvs.srv import SetBool

rclpy.init()
node = Node("gripper_client")
# TODO: create the client, wait for the service, call_async, spin until complete, print the message
node.destroy_node()
rclpy.shutdown()

In [ ]:
import time
ros.start("python gripper_server.py", log="gripper.log")
time.sleep(2)
ros.ros("python client.py close")
ros.ros("ros2 param set /gripper max_force 80.0; ros2 param get /gripper max_force")
ros.stop_all()

In [ ]:
lab.check("server")
lab.check("client")

In [ ]:
%%writefile gripper_server.cpp
// C++ (rclcpp) version of Task 1 (optional, same checker). Grade it with lab.check("server", "gripper_server.cpp"):
// the grader builds it with colcon as package rdl_gripper_server, executable gripper_server (depends on rclcpp, std_srvs, rcl_interfaces).
#include <cstdio>
#include <rclcpp/rclcpp.hpp>
#include <std_srvs/srv/set_bool.hpp>

class Gripper : public rclcpp::Node {
public:
    Gripper() : Node("gripper") {
        // TODO: declare max_force (20.0), add_on_set_parameters_callback rejecting values outside [0, 50],
        //       and the /gripper/set_closed service replying "closed with <force> N" or "opened"
    }
};

int main(int argc, char **argv) {
    rclcpp::init(argc, argv);
    rclcpp::spin(std::make_shared<Gripper>());
    rclcpp::shutdown();
}

In [ ]:
# lab.check("server", "gripper_server.cpp")  # uncomment after writing the C++ cell above

In [ ]:
%%writefile client.cpp
// C++ (rclcpp) version of Task 2 (optional, same checker). Grade it with lab.check("client", "client.cpp"):
// the grader builds it with colcon as package rdl_client, executable client (depends on rclcpp, std_srvs).
#include <chrono>
#include <cstdio>
#include <string>
#include <rclcpp/rclcpp.hpp>
#include <std_srvs/srv/set_bool.hpp>
using namespace std::chrono_literals;

int main(int argc, char **argv) {
    rclcpp::init(argc, argv);
    auto node = rclcpp::Node::make_shared("gripper_client");
    // TODO: create_client<SetBool>, wait_for_service, async_send_request (data = argv[1] == "close"),
    //       spin_until_future_complete, print the response message
    rclcpp::shutdown();
}

In [ ]:
# lab.check("client", "client.cpp")  # uncomment after writing the C++ cell above

## Checkpoint: Math
**M1 (Apply).** A service round trip takes 2 ms. A 100 Hz loop calls it every cycle. Fraction of the cycle spent waiting?

**M2 (Apply).** Ten nodes run. How many parameter services exist in total?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)

## Checkpoint: Concept
**K1 (Apply).** Pick the mechanism: (1) IMU data at 400 Hz, (2) "zero the force sensor", (3) "move the arm to the
home pose, which takes 6 s and should report progress".
- a) service, service, service
- b) topic, service, action
- c) topic, action, topic

**K2 (Evaluate).** Why validate parameters in a callback instead of checking them where they are used, and why are
parameters the wrong tool for a joint setpoint that changes every cycle?

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `ros02.h`).
3. In `roboprep/checks/ros02.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()